|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 1:</h2>|<h1>The Naive Loop<h1>|
|<h2>Section:</h2>|<h1>The arithmetic<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: size a deployment on paper<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

You are sizing a deployment. One model, three cards, and a question from
someone who wants a number: how many users at once, and how fast?

Everything here is arithmetic. No GPU, no measurement, no code that could be
slow. That is the point: you can answer this before you own the hardware.

In [ ]:
### run this cell: three real cards, from their spec sheets

cards = {
  #                bandwidth B/s   bf16 FLOP/s     VRAM bytes
  'RTX 4090':     (1008e9,         165e12,         24e9),
  'A100 80GB':    (2039e9,         312e12,         80e9),
  'H100 SXM':     (3350e9,         989e12,         80e9),
}

# and one model: Llama-3-8B
params, layers, kv_heads, head_dim = 8e9, 32, 8, 128

# Exercise 1: the ridge point of each card

FLOP per byte. The batch size at which a card stops waiting on memory.

In [ ]:
for name,(bw, flops, vram) in cards.items():
  ridge = 
  print(f'{name:12} {ridge:6.0f} FLOP/byte')

# Exercise 2: the speed limit at batch 1

One user, nobody else on the machine. A token costs exactly one read of the
weights, so this ceiling is a division and no kernel beats it.

In [ ]:
dtype_b = 2
weight_bytes = 

for name,(bw, flops, vram) in cards.items():
  # at batch 1 a token costs exactly one read of the weights
  read_ms  = 
  tok_per_s = 
  print(f'{name:12} {read_ms:6.1f} ms/token   {tok_per_s:6.0f} tok/s ceiling at batch 1')

# Exercise 3: the batch you can actually hold

Subtract the weights, divide what is left by the KV cost of one sequence at
4096 tokens, and compare that with Exercise 1.

In [ ]:
per_token = 
CTX = 4096

print(f'{"card":12} {"KV room":>9} {"fits":>6} {"ridge":>7} {"verdict"}')
for name,(bw, flops, vram) in cards.items():
  room  = 
  fits  = 
  ridge = flops / bw
  verdict = 'memory-bound' if fits < ridge else 'can reach the ridge'
  print(f'{name:12} {room/1e9:7.1f} GB {fits:6.0f} {ridge:7.0f}   {verdict}')

# Exercise 4: plot the gap

In [ ]:
ctxs = np.array([512,1024,2048,4096,8192,16384,32768])

plt.figure(figsize=(7.5,4.5))
for name,(bw, flops, vram) in cards.items():
  room = 
  plt.plot(ctxs,          , 'o-', label=name)   # sequences that fit
  plt.axhline(           , ls=':', alpha=.5)    # that card's ridge

plt.xscale('log', base=2); plt.yscale('log')
plt.xlabel('Context length'); plt.ylabel('Concurrent sequences that fit')
plt.title('Solid: what memory allows.  Dotted: what the ridge wants.')
plt.legend(); plt.grid(alpha=.3); plt.show()

### Before you open the solution

Write down an answer to each:

1. Which card has the highest ridge point? Is that the one you would want
   to serve from?
2. At 32k context, is any card able to hold the batch its ridge asks for?
3. The H100 has three times the bandwidth of the 4090. Did its ridge point
   go down, as you would hope, or up? What does that tell you about where
   hardware is heading?